# PP-2: Identify precinct neighbors
- Identify two precincts as neighbors if they share a common boundary of at least 200 feet and the edges of each precinct are within 200 feet of its neighbors’ edges. If possible, try to locate a data source for which this computation is already done.

In [2]:
import pandas as pd
import geopandas as gpd
import maup

### Load precinct GeoPackage
Reproject, reset index

In [84]:
def load_precinct_data(filepath, id_col, epsg):
    df = gpd.read_file(filepath)
    columns = [id_col, 'COUNTYFP', 'geometry']
    df = df[columns]
    
    print(df.crs)
    df = df.to_crs(epsg=epsg)
    print(df.crs)

    df = df.reset_index(drop=True)
    return df

### Identify neighboring precinct pairs 

In [85]:
def find_neighbors(df, id_col, shared_ft=200):
    sindex = df.sindex
    neighbors = []

    for i, precinct in df.iterrows():

        # Find candidates whose bounding boxes intersect (or are within 200ft)
        candidates = list(sindex.query(precinct.geometry, predicate="intersects"))
        curr_geo = df['geometry'].iloc[i]

        for j in candidates:
            if j <= i:
                continue
            
            other = df.iloc[j]
            other_geo = df['geometry'].iloc[j]

            # Compute shared boundary
            shared = precinct.geometry.boundary.intersection(other.geometry.boundary)
            shared_length = shared.length  # in feet if CRS is in feet
            
            if shared_length >= shared_ft and curr_geo.distance(other_geo) <= shared_ft:
                neighbors.append({
                    "precinct_a": precinct[id_col],
                    "precinct_b": other[id_col],
                    "shared_boundary_ft": shared_length
                })
                
    edges = pd.DataFrame(neighbors)
    print(f"Total neighbors: {len(neighbors)}")
    return edges


### Count each precinct's number of neighbors

In [86]:
def compute_neighbor_counts(edges, id_col):
    a_counts = edges["precinct_a"].value_counts()
    b_counts = edges["precinct_b"].value_counts()
    
    neighbor_counts = a_counts.add(b_counts, fill_value=0)
    neighbor_counts = neighbor_counts.reset_index()
    neighbor_counts.columns = [id_col, "num_neighbors"]
    return neighbor_counts

### Save

In [87]:
def save_results(edges, neighbor_counts, edges_path, counts_path):
    neighbor_counts.to_csv(counts_path, index=False)
    edges.to_csv(edges_path, index=False)


# GA

In [88]:
ga_path = filepath="output/Georgia/seawulf_district_maup.gpkg"
ga_id_col = "UNIQUE_ID"
ga_edges_out = "output/Georgia/ga_precinct_neighbors_map.csv"
ga_counts_out = "output/Georgia/ga_precinct_neighbors_cnt.csv"

In [89]:
ga_maup= gpd.read_file(ga_path)
ga_maup.head()

,UNIQUE_ID,COUNTYFP,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,District,geometry
0,APPLING-:-1B,001,108,921,2,1031,1074.216587,102.982480,55.003760,3.618320,1235.821146,1,"MULTIPOLYGON (((375680.672 3535246.721, 375706..."
1,APPLING-:-1C,001,67,724,0,791,1191.912652,472.673691,42.287709,28.713142,1735.587193,1,"MULTIPOLYGON (((378693.089 3524550.847, 378692..."
2,APPLING-:-2,001,782,541,4,1327,1235.612661,788.224241,96.849558,32.601383,2153.287844,1,"MULTIPOLYGON (((380005.78 3525000.183, 380113...."
3,APPLING-:-3A1,001,31,617,0,648,705.543283,121.434717,146.434230,7.825296,981.237527,1,"MULTIPOLYGON (((377772.229 3534595.104, 377753..."
4,APPLING-:-3C,001,246,934,3,1183,1427.934438,416.186299,57.464584,60.117966,1961.703287,1,"MULTIPOLYGON (((386430.886 3520109.322, 386469..."


In [90]:
ga_df = load_precinct_data(filepath = ga_path, id_col=ga_id_col, epsg = 2239)
ga_df.head()

EPSG:32617
EPSG:2239


,UNIQUE_ID,COUNTYFP,geometry
0,APPLING-:-1B,001,"MULTIPOLYGON (((610048.922 707940.202, 610132...."
1,APPLING-:-1C,001,"MULTIPOLYGON (((620310.393 672953.151, 620316...."
2,APPLING-:-2,001,"MULTIPOLYGON (((624601.545 674473.731, 624953...."
3,APPLING-:-3A1,001,"MULTIPOLYGON (((616934.398 705876.179, 616873...."
4,APPLING-:-3C,001,"MULTIPOLYGON (((645855.195 658652.878, 645980...."


In [91]:
ga_edges = find_neighbors(ga_df, ga_id_col)
ga_edges.head()

Total neighbors: 7687


,precinct_a,precinct_b,shared_boundary_ft
0,APPLING-:-1B,APPLING-:-2,5224.186963
1,APPLING-:-1B,APPLING-:-1C,66404.865904
2,APPLING-:-1B,APPLING-:-3A1,50194.563951
3,APPLING-:-1B,JEFF DAVIS-:-ALTAMAHA 2,34852.129288
4,APPLING-:-1B,TOOMBS-:-43 CEDAR CROSSING,69146.164428


In [57]:
ga_neighbor_counts = compute_neighbor_counts(ga_edges, ga_id_col)
ga_neighbor_counts.head()

,UNIQUE_ID,num_neighbors
0,APPLING-:-1B,6.0
1,APPLING-:-1C,6.0
2,APPLING-:-2,5.0
3,APPLING-:-3A1,7.0
4,APPLING-:-3C,7.0


In [ ]:
save_results(ga_edges, ga_neighbor_counts, ga_edges_out, ga_counts_out)

# AR

In [71]:
ar_path="output/Arkansas/ar_seawulf_maup.gpkg"
ar_id_col = "Unique_ID"
ar_edges_out = "output/Arkansas/ar_precinct_neighbors_map.csv"
ar_counts_out = "output/Arkansas/ar_precinct_neighbors_cnt.csv"

In [72]:
ar_maup= gpd.read_file(ar_path)
ar_maup.head()

,Unique_ID,COUNTYFP,Kamala D. Harris,Donald J. Trump,Other_candidates,Total_votes,White_population,Black_population,Latino_population,Other_population,Total_population,District,geometry
0,05001-81 - Stuttgart 2,001,293,639,19,951,486.169395,356.604974,32.728840,0.738939,876.242147,1,"MULTIPOLYGON (((2195716.192 30309.368, 2195879..."
1,05001-36 - Gillett Ward 3,001,25,39,2,66,63.524419,10.565301,0.982819,2.213607,77.286145,1,"POLYGON ((2215708.638 -7028.264, 2215708.777 -..."
2,05001-53 - Dewitt 2,001,49,188,5,242,230.348940,133.055168,0.119868,4.781286,368.305262,1,"POLYGON ((2217343.007 10978.738, 2217328.353 1..."
3,05001-51 - DeWitt 1,001,101,61,5,167,332.713262,201.772191,0.000000,7.098115,541.583568,1,"POLYGON ((2218354.249 12107.388, 2218342.578 1..."
4,05001-55 - Dewitt 3,001,42,272,3,317,36.167855,9.010094,0.713321,0.281977,46.173247,1,"POLYGON ((2216959.397 14106.194, 2216978.998 1..."


In [73]:
ar_df = load_precinct_data(filepath=ar_path, id_col=ar_id_col, epsg = 6411)
ar_df.head()

EPSG:26954
EPSG:6411


,Unique_ID,COUNTYFP,geometry
0,05001-81 - Stuttgart 2,001,"MULTIPOLYGON (((1450678.065 51999.416, 1451217..."
1,05001-36 - Gillett Ward 3,001,"POLYGON ((1497202.613 -78466.218, 1497203.072 ..."
2,05001-53 - Dewitt 2,001,"POLYGON ((1511237.47 -21029.682, 1511210.706 -..."
3,05001-51 - DeWitt 1,001,"POLYGON ((1515056.425 -17871.073, 1515062.256 ..."
4,05001-55 - Dewitt 3,001,"POLYGON ((1511516.84 -10728.805, 1511524.632 -..."


In [74]:
ar_edges = find_neighbors(ar_df, ar_id_col)
ar_edges.head()

Total neighbors: 7260


,precinct_a,precinct_b,shared_boundary_ft
0,05001-81 - Stuttgart 2,05001-41 - Morris,25488.828320
1,05001-81 - Stuttgart 2,05001-91 - Stuttgart 3,12281.342283
2,05001-81 - Stuttgart 2,05001-71 - Stuttgart 1,6494.174373
3,05001-81 - Stuttgart 2,05001-42 - Gum Pond,12601.386044
4,05001-36 - Gillett Ward 3,05001-35 - Gillett Ward 2,6071.462424


In [58]:
ar_neighbor_counts = compute_neighbor_counts(ar_edges, ar_id_col)
ar_neighbor_counts.head()

,Unique_ID,num_neighbors
0,05001-11 - McFall,5.0
1,05001-12 - Crockett,3.0
2,05001-13 - Keaton,8.0
3,05001-14 - Mill Bayou,7.0
4,05001-15 - Almyra,1.0


In [59]:
save_results(ar_edges, ar_neighbor_counts, ar_edges_out, ar_counts_out)